# Web Scraping: Crawling

---
## Imports et configuration

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
url = "https://www.allocine.fr/personne/fichepersonne-30367/filmographie/"
html = requests.get(url)
soup = BeautifulSoup(html.text, 'html.parser')
# Nous filtrons maintenant uniquement sur les films dont l'artiste est le réalisateur :
soup_director = soup.find('div', {"class" : "gd-col-left"})
# Puis nous cherchons tous les liens contenus dans cet encadré :
links = soup_director.find_all('a')

---
## Fonction final

In [20]:
def url_film1(film):
    # boucle sur le nombre de blocs html
    for i in range(len(soup.find('div', {"class" : "gd-col-left"}))):
        # Si le film est dans le bloc `title`
        if film in links[i]["title"]:
            # On renvoie le bout d'url
            resultat = links[i]["href"]
            # Break pour éviter les doublons
            break
        else:
            # Sinon rien
            None
    # Définition d'une liste pour recevoir les urls des acteurs/actrices        
    results = list()
    # scrap du bloc
    url1 = "https://www.allocine.fr" + resultat
    html1 = requests.get(url1)
    soup1 = BeautifulSoup(html1.text, 'html.parser')
    detail = soup1.find_all("a", {"class":"meta-title-link"})
    # Dans la classe meta-title-link il y a des urls polluant avec intiulé `article
    # Boucle pour gérer les bloc polluant
    for x in range(len(soup1.find_all("a", {"class":"meta-title-link"}))):
        if 'article' not in detail[x]['href']:
            # Introduction des résulat dans la liste
            results.append(detail[x]['href'])
    # Définition d'une liste pour recevoir les âge des acteurs actrices
    age_list = list()
    # Boucle sur le nombre d'acteurs/actrices
    for x in range(len(results)):
        # scrap du bloc
        url2 = "https://www.allocine.fr" + results[x]
        html2 = requests.get(url2)
        soup2 = BeautifulSoup(html2.text, 'html.parser')
        redetail = soup2.find_all("div", {"class":"meta-body-item"})
        # Insertion des âge qui se trouve à la dernière position de la liste regex
        age_list.append(int(re.findall("(\\d+)", str(redetail))[-1]))
    # renvoie de la moyenne d'âge des acteurs/actrices
    return print(f"L'age moyen des acteurs est de: {int(sum(age_list)/len(age_list))} ans")

In [21]:
url_film1("Interstellar: IMAX Exclusive")

L'age moyen des acteurs est de: 56 ans


---
## Méthodologie

In [2]:
# Fonction pour récupérer l'url du film choisi
def url_film(film):
    # boucle sur le nombre de blocs html
    for i in range(len(soup.find('div', {"class" : "gd-col-left"}))):
        # Si le film est dans le bloc `title`
        if film in links[i]["title"]:
            # On renvoie le bout d'url
            return links[i]["href"]
            # Break pour éviter les doublons
            break
        else:
            # Sinon rien
            None

In [3]:
# Appel de la fonction
resultat = url_film("Interstellar: IMAX Exclusive")
# Affichage du résultat
resultat

'/film/fichefilm_gen_cfilm=1000001104.html'

In [ ]:
# On cherche l'url des acteurs du film choisi avec un scrap sur le bloc <"a", {"class":"meta-title-link"}>
url = "https://www.allocine.fr" + resultat
html = requests.get(url)
soup = BeautifulSoup(html.text, 'html.parser')
detail = soup.find_all("a", {"class":"meta-title-link"})

In [23]:
# On obtient l'url des acteurs 
print(detail[0]['href'])

/personne/fichepersonne_gen_cpersonne=19334.html


In [26]:
# On cherche l'url des acteurs du film choisi avec un scrap sur le bloc <"a", {"class":"meta-title-link"}
# On défini une liste de réception des urls
results = list()
# Scrapping sur les blocs ("a", {"class":"meta-title-link"})
url = "https://www.allocine.fr" + resultat
html = requests.get(url)
soup = BeautifulSoup(html.text, 'html.parser')
detail = soup.find_all("a", {"class":"meta-title-link"})
# Suivi d'une boucle pour gérer les scrap l'insertion dans la liste
for x in range(len(soup.find_all("a", {"class":"meta-title-link"}))):
        results.append(detail[x]['href'])
results
# Le resultat donne des urls polluant.

['/personne/fichepersonne_gen_cpersonne=19334.html',
 '/personne/fichepersonne_gen_cpersonne=65719.html',
 '/personne/fichepersonne_gen_cpersonne=117304.html',
 '/personne/fichepersonne_gen_cpersonne=15021.html',
 '/article/fichearticle_gen_carticle=1000206611.html',
 '/article/fichearticle_gen_carticle=1000206686.html',
 '/article/fichearticle_gen_carticle=1000204569.html',
 '/article/fichearticle_gen_carticle=1000205908.html',
 '/article/fichearticle_gen_carticle=1000204967.html',
 '/article/fichearticle_gen_carticle=1000205824.html']

In [29]:
results = list()
url = "https://www.allocine.fr" + resultat
html = requests.get(url)
soup = BeautifulSoup(html.text, 'html.parser')
detail = soup.find_all("a", {"class":"meta-title-link"})
for x in range(len(soup.find_all("a", {"class":"meta-title-link"}))):
    # Condition pour la gestion des urls polluant
    # Si le terme `article` n'est pas présent dans le scrap d'url
    if 'article' not in detail[x]['href']:
        results.append(detail[x]['href'])
results
# On obtient bien les urls souhaité

['/personne/fichepersonne_gen_cpersonne=19334.html',
 '/personne/fichepersonne_gen_cpersonne=65719.html',
 '/personne/fichepersonne_gen_cpersonne=117304.html',
 '/personne/fichepersonne_gen_cpersonne=15021.html']

In [35]:
# On scrap les chiffres sur la page dans les blocs ("div", {"class":"meta-body-item"})
# Boucle sur le nombre d'acteurs/actrices (urls récupérer)
for x in range(len(results)):
    url = "https://www.allocine.fr" + results[x]
    html = requests.get(url)
    soup = BeautifulSoup(html.text, 'html.parser')
    redetail = soup.find_all("div", {"class":"meta-body-item"})
    # On fait un regex pour aller charcher les nombres dans les blocs correspondant
    m = re.findall("(\\d+)", str(redetail))
# l'âge rechercher est à la dernière position
m

['3',
 '5',
 '2',
 '2',
 '29',
 '9',
 '3',
 '5',
 '2',
 '2',
 '0',
 '0',
 '0',
 '8',
 '11',
 '1950',
 '76']

In [22]:
# Définition d'une liste pour récupérer les âges des acteurs/actrices
age_list = list()
for x in range(len(results)):
    url = "https://www.allocine.fr" + results[x]
    html = requests.get(url)
    soup = BeautifulSoup(html.text, 'html.parser')
    redetail = soup.find_all("div", {"class":"meta-body-item"})
    # Récupération des âges dans la liste (à la dernière position), en `int` pour pouvoir les calculer
    age_list.append(int(re.findall("(\\d+)", str(redetail))[-1]))
age_list

[56, 43, 49, 76]

In [6]:
# Calcul de l'age moyen : Somme des âges divisé par le bombres d'acteurs/actrices
print(f"L'age moyen des acteurs est de: {int(sum(age_list)/len(age_list))} ans")

L'age moyen des acteurs est de: 56 ans
